In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
N_PEERS = 300
N_ITERS = 5
RESULTS_PATH = "results"
T_BEGIN = 69
T_END = 170

In [ ]:
def parse_ifstat_logs():
    file_paths = [f'../{RESULTS_PATH}/{N_PEERS}/{seed}/ifstat_h{i+1}.log' for seed in range(N_ITERS) for i in range(N_PEERS)]

    raw_data = []  # This will contain N lists (one per file)
    for file_path in file_paths:
        file_sums = []  # Sum of the two floats per line
        with open(file_path, 'r') as f:
            # Skip first two title lines
            next(f, None)
            next(f, None)

            for line in f:
                parts = line.strip().split()
                
                val1 = float(parts[0])
                val2 = float(parts[1])
                file_sums.append(val1 + val2)
        raw_data.append(file_sums)

    min_length = min(len(sums) for sums in raw_data)
    return [entry[:min_length] for entry in raw_data]

In [ ]:
def main():
    # Plot each list as a separate line
    window = 10

    data = parse_ifstat_logs()

    for host_data in data:
        x_values = [(0.1 * i) - 11 for i in range(len(host_data))][:(1-window)]
        plt.plot(
            x_values, 
            np.convolve(host_data, np.ones(window)/window, mode='valid'), 
            color='black', alpha=0.01)
        # x_values = [0.1 * idx for idx in range(len(line_data))]
        # plt.plot(x_values, line_data, color='gray', alpha=0.5)

    plt.ylim(0, 2000)
    plt.xlim(T_BEGIN, T_END)
    plt.xlabel('Time (s)')
    plt.ylabel('Network usage (kb/s)')
    plt.title('Multiple Line Graph')
    plt.grid(True)
    plt.savefig(f'../{RESULTS_PATH}/{N_PEERS}_n_plot.jpg', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
for path in [f'continuous/{pkg}' for pkg in ['client-server', 'dev-eval-trickle', 'dev-v2']]:
    for n_peers in [110, 130, 200]:
        N_PEERS = n_peers
        RESULTS_PATH = path
        main()